In [50]:
# import os, shutil
# working_dir = "/kaggle/working"
# for item in os.listdir(working_dir):
#     item_path = os.path.join(working_dir, item)
#     try:
#         if os.path.isfile(item_path) or os.path.islink(item_path): os.remove(item_path)
#         elif os.path.isdir(item_path): shutil.rmtree(item_path)
#     except Exception as e: print(f"Failed to delete {item_path}: {e}")
# print("✅ Working directory cleaned.")

In [51]:
import pickle
import pandas as pd

pkl_path = "/kaggle/input/datasets/arunjatothu/web-context-embedding/web_context_embeddings.pkl"

# Load the pkl file
with open(pkl_path, "rb") as f:
    data = pickle.load(f)

# Check type of object stored
print("Type of object:", type(data))


# -------- IF DATA IS A DATAFRAME --------
if isinstance(data, pd.DataFrame):
    print("\nNumber of rows:", len(data))
    print("Columns:", data.columns.tolist())

    # Optional: check last few rows
    print("\nLast 5 rows:")
    print(data.tail())


# -------- IF DATA IS A DICTIONARY --------
elif isinstance(data, dict):
    print("\nNumber of keys:", len(data))

    # Show first few keys
    print("\nFirst 10 keys:")
    print(list(data.keys())[:10])

    # Optional: check one embedding shape
    first_key = next(iter(data))
    print("\nExample key:", first_key)

    try:
        print("Embedding shape:", data[first_key].shape)
    except:
        print("Value type:", type(data[first_key]))


# -------- IF DATA IS A LIST --------
elif isinstance(data, list):
    print("\nNumber of items:", len(data))

    print("\nFirst item:")
    print(data[0])


# -------- OTHER TYPES --------
else:
    print("\nObject content preview:")
    print(data)

Type of object: <class 'dict'>

Number of keys: 12140

First 10 keys:
['13894.png', '23058.png', '42953.png', '37408.png', '82403.png', '16952.png', '76932.png', '70914.png', '02973.png', '34096.png']

Example key: 13894.png
Embedding shape: (768,)


In [52]:
import pickle
import pandas as pd

pkl_path = "/kaggle/input/datasets/arunjatothu/web-context-embedding/web_context_embeddings.pkl"

# Load the pickle file
with open(pkl_path, "rb") as f:
    data = pickle.load(f)

print("Type:", type(data))


# -------- IF DATA IS A DICTIONARY --------
if isinstance(data, dict):

    print("\nTotal keys:", len(data))

    print("\nFirst 5 keys and values:\n")

    for i, (key, value) in enumerate(data.items()):
        if i >= 5:
            break

        print(f"Key {i+1}: {key}")

        # If embedding/vector
        try:
            print("Value shape:", value.shape)
            print("First 5 values:", value[:])
        except:
            print("Value:", value)

        print("-" * 50)


# -------- IF DATA IS A DATAFRAME --------
elif isinstance(data, pd.DataFrame):

    print("\nNumber of rows:", len(data))

    print("\nFirst 5 rows:\n")
    print(data.head())

    print("\nFirst row values:\n")
    print(data.iloc[0].values[:])


# -------- IF DATA IS A LIST --------
elif isinstance(data, list):

    print("\nNumber of items:", len(data))

    print("\nFirst 5 items:\n")

    for i in range(min(5, len(data))):
        print(f"Item {i+1}:")
        print(data[i])
        print("-" * 50)


# -------- OTHER TYPES --------
else:
    print("\nData preview:")
    print(data)

Type: <class 'dict'>

Total keys: 12140

First 5 keys and values:

Key 1: 13894.png
Value shape: (768,)
First 5 values: [ 8.52973107e-03  2.32864004e-02 -1.07110562e-02  1.00322336e-01
  2.07450241e-02  2.50629093e-02  7.97816962e-02  4.49120114e-03
 -6.92289770e-02 -1.43803041e-02 -1.40941683e-02  5.24153607e-03
  2.19555266e-04  3.53680812e-02 -8.64215009e-03 -2.04965379e-02
 -1.75123923e-02  4.98199044e-03 -3.58908088e-03  2.36717444e-02
 -5.63749373e-02  2.21887846e-02  1.33226383e-02 -1.97327025e-02
  1.94047950e-02 -3.03755384e-02 -2.10603811e-02  1.40057094e-02
 -6.47352031e-03 -2.83000823e-02 -4.97157406e-03 -9.36165545e-03
 -3.83385383e-02 -1.61751974e-02  2.32932734e-06 -2.88661718e-02
  3.12849991e-02  5.31084370e-04 -2.79521234e-02 -3.22612263e-02
  1.95294209e-02  1.08374991e-02 -7.18138367e-02  1.88533124e-02
 -3.98790874e-02  6.16169795e-02  2.11576596e-02 -1.15357712e-02
 -3.23318243e-02  2.62838490e-02 -1.73297022e-02 -1.17272418e-02
 -6.19252138e-02 -5.50325289e-02  1

In [53]:
# import gc, torch
# torch.cuda.empty_cache(); gc.collect(); torch.cuda.ipc_collect()
# print("✅ GPU memory cleared.")

In [54]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install(
    "ftfy", "regex", "tqdm",
    "git+https://github.com/openai/CLIP.git",
    "pytorch-lightning==2.2.4",
    "torchmetrics==1.3.2",
    "scikit-learn",
    "transformers>=4.35.0",
    "peft>=0.10.0",
    "pytesseract",
    "Pillow",
    "easyocr",
    "seaborn",
    "sentence-transformers>=2.6.0",   # NEW: for web-context branch
)
print("✅ All packages installed.")

✅ All packages installed.


In [55]:
import os, json, random, warnings, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from pathlib import Path
from tqdm import tqdm

import clip
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning import Trainer, seed_everything
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from torch.utils.data import Dataset, DataLoader

# LiLT imports
from transformers import LiltModel, RobertaTokenizerFast
from peft import LoraConfig, get_peft_model, TaskType

from torchvision import transforms as T
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, precision_recall_curve,
    average_precision_score, precision_recall_fscore_support,
)

warnings.filterwarnings("ignore")
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {device}")
if device == "cuda":
    print(f"   GPU : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

🖥️  Device: cuda
   GPU : Tesla T4
   VRAM: 15.6 GB


In [56]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    # ── Paths ─────────────────────────────────────────────────────────────
    dataset_root:        str  = "/kaggle/input/datasets/williamberrios/hateful-memes"
    phi_weights_path:    str  = "/kaggle/input/datasets/arunjatothu/phi-imagenet-45-2/phi_imagenet_45.pt"
    proj_weights_path:   str  = "/kaggle/input/datasets/arunjatothu/hmc-1024-projection-embeddings/hmc_1024_projection_embeddings.pt"
    lilt_model:          str  = "SCUT-DLVCLab/lilt-roberta-en-base"
    output_dir:          str  = "/kaggle/working/outputs"
    checkpoint_dir:      str  = "/kaggle/working/checkpoints"
    experiment_log_dir:  str  = "/kaggle/working/experiments"

    # ── NEW: Web-context branch paths ─────────────────────────────────────
    # Point these to the Kaggle dataset created by web_context_dataset_generation.ipynb
    web_context_emb_path: str = "/kaggle/input/datasets/arunjatothu/web-context-embedding/web_context_embeddings.pkl"
    use_web_context:      bool = True    # set False to disable the branch entirely
    web_context_dim:      int  = 768     # all-mpnet-base-v2 output dim
    web_context_proj_dim: int  = 256     # projected dim (matches layout_dim)
    web_context_dropout:  float = 0.4

    # ── Data ──────────────────────────────────────────────────────────────
    dataset:             str  = "hmc"
    image_size:          int  = 224
    fast_process:        bool = False

    # ── Model ─────────────────────────────────────────────────────────────
    name:                str  = "issues-layout"
    num_mapping_layers:  int  = 1
    map_dim:             int  = 1024
    fusion:              str  = "align"
    num_pre_output_layers: int = 1
    drop_probs:          List[float] = field(default_factory=lambda: [0.3, 0.5, 0.5])
    proj_map:            bool = False
    pretrained_proj_weights: bool = False
    freeze_proj_layers:  bool = False
    comb_proj:           bool = False
    comb_fusion:         str  = "align"
    convex_tensor:       bool = False
    text_inv_proj:       bool = False
    phi_inv_proj:        bool = False
    post_inv_proj:       bool = True
    enh_text:            bool = True
    phi_freeze:          bool = True

    pos_weight:          float = 2.0

    # ── LiLT-specific LoRA config ──────────────────────────────────────────
    lora_r:              int   = 4
    lora_alpha:          int   = 8
    lora_dropout:        float = 0.05
    lora_target_modules: List[str] = field(
        default_factory=lambda: ["layout_query", "layout_value"])

    layout_dim:          int   = 256
    layout_dropout:      float = 0.5
    use_ocr_cache:       bool  = True
    ocr_cache_dir:       str   = "/kaggle/working/ocr_cache"
    lilt_max_length:     int   = 512

    # ── Training ──────────────────────────────────────────────────────────
    batch_size:          int   = 10
    lr:                  float = 3e-5
    weight_decay:        float = 1e-3
    max_epochs:          int   = 20
    gradient_clip_val:   float = 1.0
    label_smoothing:     float = 0.2
    log_every_n_steps:   int   = 50
    val_check_interval:  float = 1.0
    limit_train_batches: float = 1.0
    limit_val_batches:   float = 1.0
    experiment_name:     str   = "issues_lilt_webctx_v1"

args = Config()

for d in [args.output_dir, args.checkpoint_dir, args.experiment_log_dir, args.ocr_cache_dir]:
    os.makedirs(d, exist_ok=True)

print("✅ Configuration loaded.")
for k, v in vars(args).items():
    print(f"   {k:35s}: {v}")

✅ Configuration loaded.
   dataset_root                       : /kaggle/input/datasets/williamberrios/hateful-memes
   phi_weights_path                   : /kaggle/input/datasets/arunjatothu/phi-imagenet-45-2/phi_imagenet_45.pt
   proj_weights_path                  : /kaggle/input/datasets/arunjatothu/hmc-1024-projection-embeddings/hmc_1024_projection_embeddings.pt
   lilt_model                         : SCUT-DLVCLab/lilt-roberta-en-base
   output_dir                         : /kaggle/working/outputs
   checkpoint_dir                     : /kaggle/working/checkpoints
   experiment_log_dir                 : /kaggle/working/experiments
   web_context_emb_path               : /kaggle/input/datasets/arunjatothu/web-context-embedding/web_context_embeddings.pkl
   use_web_context                    : True
   web_context_dim                    : 768
   web_context_proj_dim               : 256
   web_context_dropout                : 0.4
   dataset                            : hmc
   image_size

In [57]:
class FocalBCELoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.smoothing = smoothing

    def forward(self, logits, labels):
        eps    = self.smoothing
        labels = labels * (1 - eps) + 0.5 * eps
        bce    = F.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        prob   = torch.sigmoid(logits)
        pt     = torch.where(labels >= 0.5, prob, 1 - prob)
        focal_weight = (1 - pt) ** self.gamma
        alpha_t = torch.where(labels >= 0.5,
                              torch.full_like(labels, self.alpha),
                              torch.full_like(labels, 1 - self.alpha))
        return (alpha_t * focal_weight * bce).mean()

print("✅ FocalBCELoss defined.")

✅ FocalBCELoss defined.


In [58]:
import pickle, glob, zipfile

args.ocr_cache_dir = "/kaggle/working/ocr_cache"
Path(args.ocr_cache_dir).mkdir(parents=True, exist_ok=True)

OCR_SAVE_PATH = Path("/kaggle/working/ocr_cache/ocr_full_dataset.pkl")
CHUNK_GLOB    = "/kaggle/input/datasets/arunjatothu/ocr-chunk/ocr_chunk_*.pkl"

if OCR_SAVE_PATH.exists():
    print("📂 Consolidated OCR file already exists, loading...")
    with open(OCR_SAVE_PATH, "rb") as f:
        ocr_data = pickle.load(f)
    print(f"✅ Loaded {len(ocr_data)} entries  ({OCR_SAVE_PATH.stat().st_size/1e6:.1f} MB)")
else:
    chunk_files = sorted(glob.glob(CHUNK_GLOB))
    if not chunk_files:
        raise FileNotFoundError(f"No chunk files found at {CHUNK_GLOB}")
    print(f"🔨 Merging {len(chunk_files)} chunk files...")
    ocr_data = {}
    for chunk_path in tqdm(chunk_files, desc="Merging chunks"):
        with open(chunk_path, "rb") as f:
            chunk = pickle.load(f)
        for k, v in chunk.items():
            clean_key = str(k).replace("img/", "").replace("img\\", "").split("/")[-1]
            ocr_data[clean_key] = v
    with open(OCR_SAVE_PATH, "wb") as f:
        pickle.dump(ocr_data, f)
    print(f"✅ Merged OCR saved → {OCR_SAVE_PATH}  ({len(ocr_data)} entries)")

def _get_wb(entry):
    if isinstance(entry, dict):
        return entry["words"], entry["boxes"]
    return entry[0], entry[1]

ocr_lookup = {fname: _get_wb(v) for fname, v in ocr_data.items()}
print(f"✅ ocr_lookup ready — {len(ocr_lookup)} entries")

📂 Consolidated OCR file already exists, loading...
✅ Loaded 12140 entries  (3.0 MB)
✅ ocr_lookup ready — 12140 entries


In [59]:
# ═══════════════════════════════════════════════════════════════════════
# NEW — Load pre-computed web context embeddings
# Generated by web_context_dataset_generation.ipynb
# Shape: {img_id -> np.ndarray (768,)} — all-mpnet-base-v2, unit-normed
# ═══════════════════════════════════════════════════════════════════════

import pickle
import numpy as np

web_context_lookup = {}   # always defined — empty dict = branch disabled

if args.use_web_context:
    WC_PATH = Path(args.web_context_emb_path)
    if WC_PATH.exists():
        with open(WC_PATH, "rb") as f:
            web_context_lookup = pickle.load(f)
        print(f"✅ Web context embeddings loaded — {len(web_context_lookup)} entries")

        # Sanity: check a random entry
        sample_id = next(iter(web_context_lookup))
        sample_emb = web_context_lookup[sample_id]
        print(f"   Sample id   : {sample_id}")
        print(f"   Emb shape   : {sample_emb.shape}  dtype={sample_emb.dtype}")
        print(f"   Emb L2 norm : {np.linalg.norm(sample_emb):.4f}  (should be ~1.0)")

        # Confirm dim matches config
        assert sample_emb.shape[0] == args.web_context_dim, \
            f"web_context_dim mismatch: config={args.web_context_dim}, file={sample_emb.shape[0]}"
        print(f"✅ Dim check passed ({args.web_context_dim}-d)")
    else:
        print(f"⚠️  web_context_emb_path not found: {WC_PATH}")
        print("   Disabling web context branch (use_web_context=False).")
        args.use_web_context = False
else:
    print("ℹ️  Web context branch disabled in config (use_web_context=False).")

✅ Web context embeddings loaded — 12140 entries
   Sample id   : 13894.png
   Emb shape   : (768,)  dtype=float32
   Emb L2 norm : 1.0000  (should be ~1.0)
✅ Dim check passed (768-d)


In [60]:
DATA_ROOT = Path(args.dataset_root) / "hateful_memes"
IMG_ROOT  = DATA_ROOT / "img"
LABEL_DIR = Path("/kaggle/working/labels")
LABEL_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = {
    "train":       DATA_ROOT / "train.jsonl",
    "dev_seen":    DATA_ROOT / "dev_seen.jsonl",
    "dev_unseen":  DATA_ROOT / "dev_unseen.jsonl",
    "test_seen":   DATA_ROOT / "test_seen.jsonl",
    "test_unseen": DATA_ROOT / "test_unseen.jsonl",
}

dfs = []
for split_name, jsonl_path in SPLITS.items():
    if not jsonl_path.exists():
        print(f"⚠️  Missing: {jsonl_path}")
        continue
    rows = [json.loads(l) for l in open(jsonl_path)]
    df   = pd.DataFrame(rows)
    df["split"] = split_name
    if "label" not in df.columns:
        df["label"] = -1
    dfs.append(df)
    print(f"   {split_name:15s}: {len(df):5d} samples | hateful: {(df.label==1).sum()}")

full_df = pd.concat(dfs, ignore_index=True)
if "img" in full_df.columns and "image" not in full_df.columns:
    full_df["image"] = full_df["img"].apply(
        lambda x: x.split("/")[-1] if isinstance(x, str) else x)

csv_path = LABEL_DIR / "hmc_info.csv"
full_df.to_csv(csv_path, index=False)
print(f"\n✅ Merged CSV saved → {csv_path}  ({len(full_df)} rows)")

   train          :  8500 samples | hateful: 3019
   dev_seen       :   500 samples | hateful: 247
   dev_unseen     :   540 samples | hateful: 200
   test_seen      :  1000 samples | hateful: 490
   test_unseen    :  2000 samples | hateful: 750

✅ Merged CSV saved → /kaggle/working/labels/hmc_info.csv  (12540 rows)


In [61]:
# ═══════════════════════════════════════════════════════
# Combiner  (unchanged from original ISSUES)
# ═══════════════════════════════════════════════════════

class Combiner(nn.Module):
    def __init__(self, convex_tensor, input_dim, comb_proj, comb_fusion):
        super().__init__()
        self.map_dim       = input_dim
        self.comb_proj     = comb_proj
        self.comb_fusion   = comb_fusion
        self.convex_tensor = convex_tensor

        branch_out_dim      = input_dim if convex_tensor else 1
        comb_in_dim         = input_dim
        comb_concat_out_dim = comb_in_dim

        if comb_proj:
            self.comb_image_proj = nn.Sequential(
                nn.Linear(comb_in_dim, 2*comb_in_dim), nn.ReLU(), nn.Dropout(0.5))
            self.comb_text_proj  = nn.Sequential(
                nn.Linear(comb_in_dim, 2*comb_in_dim), nn.ReLU(), nn.Dropout(0.5))
            comb_in_dim = 2 * comb_in_dim

        branch_in_dim = (2 * comb_in_dim) if comb_fusion == "concat" else comb_in_dim

        self.comb_shared_branch = nn.Sequential(
            nn.Linear(branch_in_dim, 2*branch_in_dim), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(2*branch_in_dim, branch_out_dim), nn.Sigmoid())
        self.comb_concat_branch = nn.Sequential(
            nn.Linear(branch_in_dim, 2*branch_in_dim), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(2*branch_in_dim, comb_concat_out_dim))

    def forward(self, img_projection, post_features):
        pi = self.comb_image_proj(img_projection) if self.comb_proj else img_projection
        pt = self.comb_text_proj(post_features)   if self.comb_proj else post_features
        cf = torch.cat([pi, pt], dim=1) if self.comb_fusion == "concat" else torch.mul(pi, pt)
        side    = self.comb_shared_branch(cf)
        central = self.comb_concat_branch(cf)
        return central + ((1 - side) * img_projection + side * post_features)

print("✅ Combiner defined.")

✅ Combiner defined.


In [62]:
# ═══════════════════════════════════════════════════════
# TextualInversion  (Phi weights frozen — used as-is)
# ═══════════════════════════════════════════════════════

PHI_INPUT_DIM = 768

class TextualInversion(nn.Module):
    def __init__(self, clip_model, clip_img_enc_output_dim,
                 phi_proj, text_proj, post_proj,
                 drop_probs, phi_freeze, enh_text,
                 post_dim=None, num_pre_proj_layers=1):
        super().__init__()
        self.clip_model = clip_model
        self.phi_proj   = phi_proj
        self.text_proj  = text_proj
        self.post_proj  = post_proj
        self.enh_text   = enh_text
        self.output_dim = post_dim if post_proj else clip_model.token_embedding.embedding_dim

        self.phi = nn.Sequential(
            nn.Linear(PHI_INPUT_DIM, 3072), nn.GELU(), nn.Dropout(0.5),
            nn.Linear(3072, 3072),          nn.GELU(), nn.Dropout(0.5),
            nn.Linear(3072, PHI_INPUT_DIM))

        if phi_proj:
            self.phi_map = nn.Sequential(
                nn.Linear(PHI_INPUT_DIM, PHI_INPUT_DIM), nn.Dropout(drop_probs[0]))

        phi_dict = torch.load(args.phi_weights_path, map_location="cpu")["MLPCustom"]
        with torch.no_grad():
            for i, layer_i in enumerate([0, 3, 6]):
                self.phi[layer_i].weight.copy_(phi_dict[f"layers.{layer_i}.weight"])
                self.phi[layer_i].bias.copy_(phi_dict[f"layers.{layer_i}.bias"])

        for p in self.phi.parameters():
            p.requires_grad_(False)

        layers = [nn.Linear(clip_img_enc_output_dim, PHI_INPUT_DIM), nn.Dropout(drop_probs[0])]
        for _ in range(1, num_pre_proj_layers):
            layers += [nn.ReLU(), nn.Linear(PHI_INPUT_DIM, PHI_INPUT_DIM), nn.Dropout(drop_probs[0])]
        self.pre_inversion_map = nn.Sequential(*layers)

        if post_proj:
            self.post_inversion_map = nn.Sequential(
                nn.Linear(clip_model.token_embedding.embedding_dim, post_dim),
                nn.Dropout(drop_probs[0]))

    def encode_with_vstar(self, clip_model, text, v_star, num_vstar=1,
                          pooling=True, token_id=259, proj=True):
        x = clip_model.token_embedding(text).type(clip_model.dtype)
        _, counts = torch.unique((text == token_id).nonzero(as_tuple=True)[0], return_counts=True)
        cum_sum = torch.cat((torch.zeros(1, device=x.device).int(),
                             torch.cumsum(counts, dim=0)[:-1]))
        first_vstar_indexes = (text == token_id).nonzero()[cum_sum][:, 1]
        rep_idx = torch.cat([(first_vstar_indexes + n).unsqueeze(0) for n in range(num_vstar)])
        if v_star.shape[0] == x.shape[0]:
            if len(v_star.shape) == 2: v_star = v_star.unsqueeze(1)
            x[torch.arange(x.shape[0]).repeat_interleave(num_vstar).reshape(x.shape[0], num_vstar),
              rep_idx.T] = v_star.to(x.dtype)
        x = x + clip_model.positional_embedding.type(clip_model.dtype)
        x = clip_model.transformer(x.permute(1, 0, 2)).permute(1, 0, 2)
        x = clip_model.ln_final(x).type(clip_model.dtype)
        if pooling:
            x = (x[torch.arange(x.shape[0]), text.argmax(dim=-1)] @ clip_model.text_projection
                 if proj else x[torch.arange(x.shape[0]), text.argmax(dim=-1)])
        return x

    def forward(self, prompt, image_features):
        img_f  = self.pre_inversion_map(image_features)
        v_star = self.phi(img_f)
        if self.phi_proj: v_star = self.phi_map(v_star)
        features = self.encode_with_vstar(
            self.clip_model, prompt, v_star, proj=self.text_proj).float()
        if self.post_proj: features = self.post_inversion_map(features)
        return features

print("✅ TextualInversion defined (Phi weights frozen).")

✅ TextualInversion defined (Phi weights frozen).


In [63]:
# ═══════════════════════════════════════════════════════
# LiLTLayoutBranch — replaces LayoutLMv3 LayoutBranch
# ═══════════════════════════════════════════════════════

LILT_HIDDEN = 768

class LiLTLayoutBranch(nn.Module):
    def __init__(self, model_name, output_dim, dropout,
                 lora_r, lora_alpha, lora_dropout, lora_target_modules):
        super().__init__()
        base_model = LiltModel.from_pretrained(model_name)

        n_frozen = 0
        for name, param in base_model.named_parameters():
            if "layout_" not in name:
                param.requires_grad_(False)
                n_frozen += param.numel()
        print(f"   LiLT: Froze {n_frozen/1e6:.2f}M text-side params")

        lora_cfg = LoraConfig(
            r=lora_r, lora_alpha=lora_alpha,
            target_modules=lora_target_modules,
            lora_dropout=lora_dropout, bias="none",
            task_type=TaskType.FEATURE_EXTRACTION,
        )
        self.encoder = get_peft_model(base_model, lora_cfg)

        n_trainable = sum(p.numel() for p in self.encoder.parameters() if p.requires_grad)
        n_total     = sum(p.numel() for p in self.encoder.parameters())
        print(f"   LiLTLayoutBranch (LoRA r={lora_r}): "
              f"{n_trainable/1e6:.3f}M / {n_total/1e6:.2f}M params trainable "
              f"({100*n_trainable/n_total:.2f}%)")

        self.proj = nn.Sequential(
            nn.Linear(LILT_HIDDEN, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, input_ids, attention_mask, bbox):
        outputs  = self.encoder(input_ids=input_ids,
                                attention_mask=attention_mask, bbox=bbox)
        cls_feat = outputs.last_hidden_state[:, 0, :]
        return self.proj(cls_feat)

print("✅ LiLTLayoutBranch defined (RoBERTa frozen, LoRA on layout attention only).")

✅ LiLTLayoutBranch defined (RoBERTa frozen, LoRA on layout attention only).


In [64]:
# ═══════════════════════════════════════════════════════════════════════
# NEW — WebContextBranch
#
# Takes pre-computed all-mpnet-base-v2 sentence embeddings (768-d,
# unit-normed) for each meme — derived from DuckDuckGo snippets for
# the meme's OCR text — and projects them to web_context_proj_dim.
#
# Why this helps:
#   • CLIP + LiLT see image pixels and OCR layout — they don't know
#     what the OCR text *means* in the broader world.
#   • Web context adds external knowledge: e.g. for a meme containing
#     "George Soros", the web snippets encode known context about that
#     entity, giving the model a signal about whether it's commonly
#     associated with hate-speech targets.
#   • The branch is frozen-input: mpnet embeddings are fixed, only the
#     small 2-layer projection is trained (< 0.5M params).
# ═══════════════════════════════════════════════════════════════════════

class WebContextBranch(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, dropout: float):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, output_dim * 2),
            nn.LayerNorm(output_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(output_dim * 2, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, input_dim) — pre-computed sentence embeddings, already unit-normed
        Returns:
            (B, output_dim)
        """
        return self.proj(x)

print("✅ WebContextBranch defined.")

✅ WebContextBranch defined.


In [65]:
# ═══════════════════════════════════════════════════════════════════════
# QuadModalCombiner
#
# Fuses 4 streams:
#   1. visual_feat    (map_dim)  — CLIP image projection
#   2. text_inv_feat  (map_dim)  — Phi textual inversion
#   3. layout_feat    (layout_dim) — LiLT layout branch
#   4. web_ctx_feat   (web_context_proj_dim) — web context branch [NEW]
#
# Architecture:
#   • Step 1: bimodal ISSUES combiner on (visual, text_inv) → bi (map_dim)
#   • Step 2: layout gating on (bi, layout) — same as TriModalCombiner
#   • Step 3: web-context cross-attention gate on top of step-2 output
#             The gate learns how much web context to mix in per sample.
#             High-confidence web context (entity-rich memes) gets high
#             gate weight; ambiguous web context gets low weight.
# ═══════════════════════════════════════════════════════════════════════

class QuadModalCombiner(nn.Module):
    def __init__(self, map_dim, layout_dim, web_ctx_dim,
                 comb_proj, comb_fusion, convex_tensor,
                 use_web_context=True):
        super().__init__()
        self.use_web_context = use_web_context

        # ── Step 1: bimodal (visual + text_inv) ──────────────────────────
        self.bimodal = Combiner(convex_tensor, map_dim, comb_proj, comb_fusion)

        # ── Step 2: layout gating (same as TriModalCombiner) ─────────────
        self.layout_up = nn.Sequential(
            nn.Linear(layout_dim, map_dim),
            nn.LayerNorm(map_dim),
            nn.GELU(),
            nn.Dropout(0.4),
        )
        gate_in = 2 * map_dim
        self.layout_gate = nn.Sequential(
            nn.Linear(gate_in, map_dim), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(map_dim, 1), nn.Sigmoid()
        )
        self.layout_proj = nn.Sequential(
            nn.Linear(gate_in, map_dim), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(map_dim, map_dim)
        )
        self.layout_drop = nn.Dropout(p=0.1)

        # ── Step 3: web-context gating [NEW] ─────────────────────────────
        if use_web_context:
            web_gate_in = map_dim + web_ctx_dim
            self.web_gate = nn.Sequential(
                nn.Linear(web_gate_in, map_dim // 2), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(map_dim // 2, 1), nn.Sigmoid()
            )
            self.web_proj = nn.Sequential(
                nn.Linear(web_gate_in, map_dim), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(map_dim, map_dim)
            )
            self.web_drop = nn.Dropout(p=0.1)

    def forward(self, visual_feat, text_inv_feat, layout_feat, web_ctx_feat=None):
        # Step 1: bimodal
        bi = self.bimodal(visual_feat, text_inv_feat)

        # Step 2: layout gate
        lf  = self.layout_up(layout_feat)
        cat = torch.cat([bi, lf], dim=1)
        g   = self.layout_gate(cat)
        lc  = self.layout_proj(cat)
        fused = (1 - g) * bi + g * self.layout_drop(lc)

        # Step 3: web-context gate [NEW]
        if self.use_web_context and web_ctx_feat is not None:
            wcat = torch.cat([fused, web_ctx_feat], dim=1)
            wg   = self.web_gate(wcat)
            wc   = self.web_proj(wcat)
            fused = (1 - wg) * fused + wg * self.web_drop(wc)

        return fused

print("✅ QuadModalCombiner defined.")

✅ QuadModalCombiner defined.


In [66]:
class MemesDataset(Dataset):
    def __init__(self, data_root, split="train", image_size=224,
                 fast=False, ocr_lookup=None, web_context_lookup=None):
        super().__init__()
        self.data_root          = Path(data_root) / "hateful_memes"
        self.img_root           = self.data_root / "img"
        self.split              = split
        self.image_size         = image_size
        self.ocr_lookup         = ocr_lookup or {}
        self.web_context_lookup = web_context_lookup or {}

        csv_path = Path("/kaggle/working/labels/hmc_info.csv")
        self.df  = pd.read_csv(csv_path)
        self.df  = self.df[self.df["split"] == split].reset_index(drop=True)
        float_cols = self.df.select_dtypes(float).columns
        self.df[float_cols] = self.df[float_cols].fillna(-1).astype("Int64")

        if self.ocr_lookup:
            missing = [str(fn).split("/")[-1]
                       for fn in self.df["image"]
                       if str(fn).split("/")[-1] not in self.ocr_lookup]
            if missing:
                print(f"  ⚠️  [{split}] {len(missing)} images not in ocr_lookup — [UNK] fallback")

        # NEW: report web context coverage
        if self.web_context_lookup:
            wc_missing = [str(fn).split("/")[-1]
                          for fn in self.df["image"]
                          if str(fn).split("/")[-1] not in self.web_context_lookup]
            if wc_missing:
                print(f"  ⚠️  [{split}] {len(wc_missing)} images not in web_context_lookup — mean-vec fallback")

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        txt    = "null" if str(row.get("text", "")).lower() == "nothing" else str(row.get("text", ""))
        img_fn = row.get("image", row.get("img", ""))
        if isinstance(img_fn, str) and "/" in img_fn:
            img_fn = img_fn.split("/")[-1]
        image  = Image.open(self.img_root / img_fn).convert("RGB").resize((self.image_size, self.image_size))
        words, boxes = self.ocr_lookup.get(str(img_fn), (["[UNK]"], [[0, 0, 1000, 1000]]))

        # NEW: fetch web context embedding (numpy array 768-d, or None)
        web_ctx = self.web_context_lookup.get(str(img_fn), None)

        return {
            "image":       image,
            "text":        txt,
            "label":       int(row.get("label", -1)),
            "idx_meme":    int(row.get("id", idx)),
            "origin_text": txt,
            "img_path":    str(self.img_root / img_fn),
            "ocr_words":   words,
            "ocr_boxes":   boxes,
            "is_train":    self.split == "train",
            "web_ctx":     web_ctx,   # NEW: np.ndarray (768,) or None
        }


class MemesCollator:
    def __init__(self, args, web_context_lookup=None):
        self.args               = args
        self.web_context_lookup = web_context_lookup or {}
        _, self.clip_preprocess = clip.load("ViT-L/14", device="cpu", jit=False)
        self._lilt_tokenizer    = None
        self.train_aug          = T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.0),
            T.RandomGrayscale(p=0.05),
        ])

        # Pre-compute global mean embedding as fallback for missing web context
        if self.web_context_lookup:
            all_embs = np.stack(list(self.web_context_lookup.values()))
            mean_emb = all_embs.mean(axis=0).astype(np.float32)
            mean_emb /= (np.linalg.norm(mean_emb) + 1e-8)
            self._web_mean = torch.from_numpy(mean_emb)
        else:
            self._web_mean = torch.zeros(args.web_context_dim)

    @property
    def lilt_tokenizer(self):
        if self._lilt_tokenizer is None:
            self._lilt_tokenizer = RobertaTokenizerFast.from_pretrained(
                args.lilt_model, add_prefix_space=True)
        return self._lilt_tokenizer

    def _encode_lilt(self, batch_words, batch_boxes, max_length=512):
        tok = self.lilt_tokenizer
        all_input_ids      = []
        all_attention_mask = []
        all_bbox           = []
        for words, boxes in zip(batch_words, batch_boxes):
            if not words or words == ["[UNK]"]:
                words = ["meme"]; boxes = [[0, 0, 1000, 1000]]
            boxes = [[max(0, min(1000, c)) for c in b] for b in boxes]
            enc = tok(words, is_split_into_words=True,
                      max_length=max_length, padding="max_length",
                      truncation=True, return_tensors="pt")
            word_ids    = enc.word_ids(batch_index=0)
            aligned_box = []
            for wid in word_ids:
                aligned_box.append([0, 0, 0, 0] if wid is None else boxes[wid])
            all_input_ids.append(enc["input_ids"])
            all_attention_mask.append(enc["attention_mask"])
            all_bbox.append(torch.tensor([aligned_box], dtype=torch.long))
        return {
            "input_ids":      torch.cat(all_input_ids,      dim=0),
            "attention_mask": torch.cat(all_attention_mask, dim=0),
            "bbox":           torch.cat(all_bbox,           dim=0),
        }

    def __call__(self, batch):
        labels    = torch.LongTensor([item["label"]    for item in batch])
        idx_memes = torch.LongTensor([item["idx_meme"] for item in batch])
        img_paths = [item["img_path"] for item in batch]

        enh_texts = torch.cat([
            clip.tokenize(f'a photo of $ , {el["origin_text"]}',
                          context_length=77, truncate=True)
            for el in batch], dim=0)
        simple_prompt = clip.tokenize(
            "a photo of $", context_length=77).repeat(labels.shape[0], 1)

        img = torch.cat([
            self.clip_preprocess(
                self.train_aug(item["image"]) if item.get("is_train") else item["image"]
            ).unsqueeze(0)
            for item in batch], dim=0)

        texts = torch.cat([clip.tokenize(item["text"], context_length=77, truncate=True)
                           for item in batch], dim=0)

        out = {
            "labels": labels, "idx_memes": idx_memes,
            "enhanced_texts": enh_texts, "simple_prompt": simple_prompt,
            "pixel_values": img, "texts": texts, "img_paths": img_paths,
        }

        if args.name == "issues-layout":
            lilt_enc = self._encode_lilt(
                [item["ocr_words"] for item in batch],
                [item["ocr_boxes"] for item in batch],
                max_length=args.lilt_max_length,
            )
            out["layout_input_ids"]      = lilt_enc["input_ids"]
            out["layout_attention_mask"] = lilt_enc["attention_mask"]
            out["layout_bbox"]           = lilt_enc["bbox"]

        # NEW: collate web context embeddings into a (B, 768) tensor
        if args.use_web_context:
            web_embs = []
            for item in batch:
                emb = item.get("web_ctx", None)
                if emb is not None:
                    web_embs.append(torch.from_numpy(emb.astype(np.float32)))
                else:
                    web_embs.append(self._web_mean.clone())
            out["web_ctx"] = torch.stack(web_embs, dim=0)   # (B, 768)

        return out

print("✅ MemesDataset & MemesCollator (with web context) defined.")

✅ MemesDataset & MemesCollator (with web context) defined.


In [67]:
class EpochLogger(pl.Callback):
    COLS = [
        ("epoch",      "Ep",       5),
        ("train_loss", "TrLoss",   8),
        ("val_loss",   "VaLoss",   8),
        ("train_acc",  "TrAcc",    7),
        ("val_acc",    "VaAcc",    7),
        ("train_f1",   "TrF1",     7),
        ("val_f1",     "VaF1",     7),
        ("train_auroc","TrAUROC",  9),
        ("val_auroc",  "VaAUROC",  9),
    ]

    def __init__(self, log_path, experiment_name):
        super().__init__()
        self.log_path        = log_path
        self.experiment_name = experiment_name
        self.records         = []

    @staticmethod
    def _f(val): return f"{val:.4f}" if val == val else "  —   "

    def _header(self):
        parts = [f"{h:^{w}}" for _, h, w in self.COLS]
        sep   = ["-"*w       for _, _, w in self.COLS]
        return "  |  ".join(parts) + "\n" + "--+--".join(sep)

    def _row(self, rec, best_epoch):
        star  = "★" if int(rec["epoch"]) == best_epoch else " "
        parts = []
        for key, _, w in self.COLS:
            val = rec.get(key, float("nan"))
            s   = (f"{int(val):^{w}}" if key == "epoch" else f"{self._f(val):^{w}}")
            parts.append(s)
        return star + " " + "  |  ".join(parts)

    def _print_table(self):
        if not self.records: return
        df = pd.DataFrame(self.records)
        
        # FIX: drop NaN before finding best epoch
        valid = df["val_auroc"].dropna()
        if valid.empty: return          # nothing to print yet
        best_ep = int(df.loc[valid.idxmax(), "epoch"])
        
        sep = "=" * (sum(w for _, _, w in self.COLS) + 5*(len(self.COLS)-1) + 2)
        tqdm.write("")
        tqdm.write(f"  📋  Epoch History — {self.experiment_name}")
        tqdm.write(sep)
        tqdm.write("  " + self._header())
        tqdm.write(sep)
        for _, row in df.iterrows():
            tqdm.write(self._row(row.to_dict(), best_ep))
        tqdm.write(sep)
        tqdm.write(f"  ★ = best val AUROC (epoch {best_ep})")
        tqdm.write("")

    def on_train_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking: return
        logs = trainer.callback_metrics
        rec  = {
            "epoch":      trainer.current_epoch + 1,
            "train_loss": float(logs.get("train/loss_epoch", float("nan"))),
            "train_acc":  float(logs.get("train/accuracy",  float("nan"))),
            "train_f1":   float(logs.get("train/f1",        float("nan"))),
            "train_auroc":float(logs.get("train/auroc",     float("nan"))),
        }
        self._partial = rec

    def on_validation_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking: return
        logs = trainer.callback_metrics
        rec  = getattr(self, "_partial", {"epoch": trainer.current_epoch + 1})
        rec.update({
            "val_loss":  float(logs.get("val/loss",     float("nan"))),
            "val_acc":   float(logs.get("val/accuracy", float("nan"))),
            "val_f1":    float(logs.get("val/f1",       float("nan"))),
            "val_auroc": float(logs.get("val/auroc",    float("nan"))),
        })
        self.records.append(rec)
        self._partial = {}    # ADD THIS
        pd.DataFrame(self.records).to_csv(self.log_path, index=False)
        self._print_table()

print("✅ EpochLogger defined.")

✅ EpochLogger defined.


In [68]:
CLIP_IMG_ENC_OUTPUT_DIM_BEFORE_PROJ = 1024

class LinearProjection(nn.Module):
    def __init__(self, input_dim, output_dim, num_layers, drop_probs):
        super().__init__()
        layers = [nn.Linear(input_dim, output_dim), nn.Dropout(drop_probs[0])]
        for _ in range(1, num_layers):
            layers += [nn.ReLU(), nn.Linear(output_dim, output_dim), nn.Dropout(drop_probs[0])]
        self.proj = nn.Sequential(*layers)
    def forward(self, x): return self.proj(x)


class HateClassifier(pl.LightningModule):
    """
    ISSUES + LiLT + WebContext (4-modal) variant.

    Frozen:    CLIP backbone, Phi weights, RoBERTa text side of LiLT
    Trainable: LoRA adapters on LiLT layout attention,
               all projections, combiners, WebContextBranch (tiny)
    """

    def __init__(self, args):
        super().__init__()
        self.args               = args
        self.dataset            = args.dataset
        self.num_mapping_layers = args.num_mapping_layers
        self.map_dim            = args.map_dim
        self.fusion             = args.fusion
        self.num_pre_output_layers = args.num_pre_output_layers
        self.lr                 = args.lr
        self.weight_decay       = args.weight_decay
        self.name               = args.name
        self.fast_process       = args.fast_process
        self.proj_map           = args.proj_map
        self.convex_tensor      = args.convex_tensor
        self.comb_proj          = args.comb_proj
        self.comb_fusion        = args.comb_fusion
        self.enh_text           = args.enh_text
        self.phi_freeze         = True
        self.text_inv_proj      = args.text_inv_proj
        self.phi_inv_proj       = args.phi_inv_proj
        self.post_inv_proj      = args.post_inv_proj
        self.label_smoothing    = args.label_smoothing
        self.use_web_context    = args.use_web_context   # NEW

        for prefix in ("train", "val", "test"):
            setattr(self, f"{prefix}_acc",   torchmetrics.Accuracy(task="binary"))
            setattr(self, f"{prefix}_auroc", torchmetrics.AUROC(task="binary"))
            setattr(self, f"{prefix}_prec",  torchmetrics.Precision(task="binary"))
            setattr(self, f"{prefix}_rec",   torchmetrics.Recall(task="binary"))
            setattr(self, f"{prefix}_f1",    torchmetrics.F1Score(task="binary"))

        # ── CLIP backbone (frozen) ─────────────────────────────────────────
        self.clip_model, _ = clip.load("ViT-L/14", device="cpu", jit=False)
        self.clip_model.visual.proj = None
        self.clip_model.float()
        for p in self.clip_model.parameters():
            p.requires_grad_(False)

        self.image_map = LinearProjection(CLIP_IMG_ENC_OUTPUT_DIM_BEFORE_PROJ,
                                          self.map_dim, self.num_mapping_layers, args.drop_probs)
        self.text_map  = LinearProjection(self.clip_model.token_embedding.embedding_dim,
                                          self.map_dim, self.num_mapping_layers, args.drop_probs)

        pre_output_input_dim = self._build_head(args)

        proj_path = getattr(args, "proj_weights_path", None)
        if proj_path and os.path.exists(proj_path):
            proj_sd = torch.load(proj_path, map_location="cpu", weights_only=False)["state_dict"]
            with torch.no_grad():
                self.image_map.proj[0].weight.copy_(proj_sd["image_proj_weight"])
                self.image_map.proj[0].bias.copy_(proj_sd["image_proj_bias"])
                self.text_map.proj[0].weight.copy_(proj_sd["text_proj_weight"])
                self.text_map.proj[0].bias.copy_(proj_sd["text_proj_bias"])
            print("✅ Loaded pre-trained ISSUES projection weights into image_map & text_map")
        else:
            print("⚠️  proj_weights_path not set — image_map & text_map initialised randomly")

        pre_output_layers = [nn.Dropout(args.drop_probs[1])]
        output_input_dim  = pre_output_input_dim
        if self.num_pre_output_layers >= 1:
            pre_output_layers += [nn.Linear(pre_output_input_dim, self.map_dim),
                                  nn.ReLU(), nn.Dropout(args.drop_probs[2])]
            output_input_dim = self.map_dim
        for _ in range(1, self.num_pre_output_layers):
            pre_output_layers += [nn.Linear(self.map_dim, self.map_dim),
                                  nn.ReLU(), nn.Dropout(args.drop_probs[2])]

        self.pre_output = nn.Sequential(*pre_output_layers)
        self.output     = nn.Linear(output_input_dim, 1)
        self.criterion  = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([args.pos_weight]), reduction="mean")

    def _build_head(self, args):
        if self.name == "issues-layout":
            self.text_inv = TextualInversion(
                self.clip_model, CLIP_IMG_ENC_OUTPUT_DIM_BEFORE_PROJ,
                args.phi_inv_proj, args.text_inv_proj, args.post_inv_proj,
                args.drop_probs, phi_freeze=True, enh_text=args.enh_text,
                post_dim=self.map_dim, num_pre_proj_layers=args.num_mapping_layers)
            self.image_map = LinearProjection(CLIP_IMG_ENC_OUTPUT_DIM_BEFORE_PROJ,
                                              self.map_dim, args.num_mapping_layers, args.drop_probs)
            self.layout_branch = LiLTLayoutBranch(
                model_name=args.lilt_model,
                output_dim=args.layout_dim,
                dropout=args.layout_dropout,
                lora_r=args.lora_r, lora_alpha=args.lora_alpha,
                lora_dropout=args.lora_dropout,
                lora_target_modules=args.lora_target_modules,
            )

            # NEW: web context branch (tiny, ~0.5M params)
            if self.use_web_context:
                self.web_ctx_branch = WebContextBranch(
                    input_dim  = args.web_context_dim,
                    output_dim = args.web_context_proj_dim,
                    dropout    = args.web_context_dropout,
                )

            # CHANGED: TriModalCombiner → QuadModalCombiner
            self.quad_combiner = QuadModalCombiner(
                map_dim          = self.map_dim,
                layout_dim       = args.layout_dim,
                web_ctx_dim      = args.web_context_proj_dim,
                comb_proj        = args.comb_proj,
                comb_fusion      = args.comb_fusion,
                convex_tensor    = args.convex_tensor,
                use_web_context  = args.use_web_context,
            )
            return self.map_dim

        elif self.name in ["hate-clipper", "adaptation"]:
            return self.map_dim if args.fusion == "align" else self.map_dim * 2
        elif self.name in ["text-only", "image-only"]:
            return (self.map_dim if self.proj_map
                    else (self.clip_model.token_embedding.embedding_dim
                          if self.name == "text-only" else CLIP_IMG_ENC_OUTPUT_DIM_BEFORE_PROJ))
        elif self.name == "sum":
            return self.map_dim
        elif self.name == "combiner":
            self.comb = Combiner(self.convex_tensor, self.map_dim, self.comb_proj, self.comb_fusion)
            return self.map_dim
        elif self.name in ("text-inv", "text-inv-comb"):
            self.text_inv = TextualInversion(
                self.clip_model, CLIP_IMG_ENC_OUTPUT_DIM_BEFORE_PROJ,
                self.phi_inv_proj, self.text_inv_proj, self.post_inv_proj,
                args.drop_probs, phi_freeze=True, enh_text=self.enh_text,
                post_dim=self.map_dim, num_pre_proj_layers=self.num_mapping_layers)
            if self.name == "text-inv-comb":
                self.comb = Combiner(self.convex_tensor, self.map_dim, self.comb_proj, self.comb_fusion)
                self.image_map = LinearProjection(CLIP_IMG_ENC_OUTPUT_DIM_BEFORE_PROJ,
                                                  self.map_dim, self.num_mapping_layers, args.drop_probs)
            return self.text_inv.output_dim if self.name == "text-inv" else self.map_dim
        raise ValueError(f"Unknown model name: {self.name}")

    def compute_CLIP_features(self, img, text):
        img_f = self.clip_model.visual(img.type(self.clip_model.dtype))
        x = self.clip_model.token_embedding(text).type(self.clip_model.dtype)
        x = x + self.clip_model.positional_embedding.type(self.clip_model.dtype)
        x = self.clip_model.transformer(x.permute(1, 0, 2)).permute(1, 0, 2)
        x = self.clip_model.ln_final(x).type(self.clip_model.dtype)
        tf = x[torch.arange(x.shape[0]), text.argmax(dim=-1)]
        return img_f, tf

    def _smooth_bce(self, logits, labels):
        return self.criterion(logits, labels.float())

    def common_step(self, batch):
        img_f, txt_f = self.compute_CLIP_features(batch["pixel_values"], batch["texts"])
        prompt = batch["enhanced_texts"] if self.enh_text else batch["simple_prompt"]
        n = self.name

        if n == "issues-layout":
            ti_feat  = self.text_inv(prompt, img_f)
            vis_feat = self.image_map(img_f)
            lay_feat = self.layout_branch(
                input_ids      = batch["layout_input_ids"].to(img_f.device),
                attention_mask = batch["layout_attention_mask"].to(img_f.device),
                bbox           = batch["layout_bbox"].to(img_f.device),
            )

            # NEW: web context forward
            web_feat = None
            if self.use_web_context and "web_ctx" in batch:
                raw = batch["web_ctx"].to(img_f.device).float()
                web_feat = self.web_ctx_branch(raw)

            features = self.quad_combiner(vis_feat, ti_feat, lay_feat, web_feat)

        elif n in ["hate-clipper", "adaptation"]:
            img_f = self.image_map(img_f); txt_f = self.text_map(txt_f)
            features = torch.mul(img_f, txt_f) if self.fusion == "align" else torch.cat([img_f, txt_f], dim=1)
        elif n == "text-only":
            features = self.text_map(txt_f) if self.proj_map else txt_f
        elif n == "image-only":
            features = self.image_map(img_f) if self.proj_map else img_f
        elif n == "sum":
            features = self.image_map(img_f) + self.text_map(txt_f)
        elif n == "combiner":
            features = self.comb(self.image_map(img_f), self.text_map(txt_f))
        elif n == "text-inv":
            features = self.text_inv(prompt, img_f)
        elif n == "text-inv-comb":
            features = self.comb(self.image_map(img_f), self.text_inv(prompt, img_f))
        else:
            raise ValueError(f"Unknown model name: {n}")

        logits      = self.output(self.pre_output(features)).squeeze(1)
        preds_proxy = torch.sigmoid(logits)
        preds       = (preds_proxy >= 0.5).long()
        return {
            "loss":        self._smooth_bce(logits, batch["labels"].float()),
            "preds":       preds,
            "preds_proxy": preds_proxy,
            "labels":      batch["labels"],
        }

    # ── Training ──────────────────────────────────────────────────────────
    def training_step(self, batch, batch_idx):
        out = self.common_step(batch)
        self.train_acc.update(out["preds"],         out["labels"])
        self.train_auroc.update(out["preds_proxy"], out["labels"])
        self.train_prec.update(out["preds"],        out["labels"])
        self.train_rec.update(out["preds"],         out["labels"])
        self.train_f1.update(out["preds"],          out["labels"])
        self.log("train/loss", out["loss"], on_step=True, on_epoch=True, prog_bar=True)
        self.log("train/acc",  self.train_acc.compute(),  on_step=True, on_epoch=False, prog_bar=True)
        self.log("train/f1",   self.train_f1.compute(),   on_step=True, on_epoch=False, prog_bar=True)
        return out["loss"]

    def on_train_epoch_end(self):
        self.log("train/accuracy",  self.train_acc.compute(),   prog_bar=True)
        self.log("train/auroc",     self.train_auroc.compute(), prog_bar=True)
        self.log("train/precision", self.train_prec.compute())
        self.log("train/recall",    self.train_rec.compute())
        self.log("train/f1",        self.train_f1.compute(),    prog_bar=True)
        for m in (self.train_acc, self.train_auroc, self.train_prec, self.train_rec, self.train_f1):
            m.reset()

    # ── Validation ────────────────────────────────────────────────────────
    def validation_step(self, batch, batch_idx):
        out = self.common_step(batch)
        self.val_acc.update(out["preds"],         out["labels"])
        self.val_auroc.update(out["preds_proxy"], out["labels"])
        self.val_prec.update(out["preds"],        out["labels"])
        self.val_rec.update(out["preds"],         out["labels"])
        self.val_f1.update(out["preds"],          out["labels"])
        self.log("val/loss", out["loss"], on_step=False, on_epoch=True, prog_bar=True)
        self.log("val/acc",  self.val_acc.compute(), prog_bar=True)
        self.log("val/f1",   self.val_f1.compute(),  prog_bar=True)
        return out

# In HateClassifier — replace the broken on_validation_epoch_end with this:
    def on_validation_epoch_end(self):
        if self.trainer.sanity_checking: return
        self.log("val/accuracy",  self.val_acc.compute(),   prog_bar=True)
        self.log("val/auroc",     self.val_auroc.compute(), prog_bar=True)
        self.log("val/precision", self.val_prec.compute())
        self.log("val/recall",    self.val_rec.compute())
        self.log("val/f1",        self.val_f1.compute(),    prog_bar=True)
        for m in (self.val_acc, self.val_auroc, self.val_prec, self.val_rec, self.val_f1):
            m.reset()

    # ── Test ──────────────────────────────────────────────────────────────
    def test_step(self, batch, batch_idx, dataloader_idx=0):
        prefix = {0: "dev_seen", 1: "test_seen", 2: "dev_unseen", 3: "test_unseen"}.get(dataloader_idx, "test")
        out = self.common_step(batch)
        self.test_acc.update(out["preds"],         out["labels"])
        self.test_auroc.update(out["preds_proxy"], out["labels"])
        self.log(f"{prefix}/accuracy", self.test_acc.compute())
        self.log(f"{prefix}/auroc",    self.test_auroc.compute())
        return out

    # ── Optimiser ─────────────────────────────────────────────────────────
    def configure_optimizers(self):
        lora_params  = []
        other_params = []
        no_decay     = ["bias", "LayerNorm.weight", "layer_norm.weight"]

        for name, p in self.named_parameters():
            if not p.requires_grad: continue
            if "layout_branch.encoder" in name:
                lora_params.append(p)
            else:
                other_params.append((name, p))

        param_groups = [
            {"params": [p for n, p in other_params if not any(nd in n for nd in no_decay)],
             "lr": self.lr, "weight_decay": self.weight_decay, "name": "scratch_decay"},
            {"params": [p for n, p in other_params if any(nd in n for nd in no_decay)],
             "lr": self.lr, "weight_decay": 0.0, "name": "scratch_nodecay"},
            {"params": lora_params, "lr": self.lr * 0.1,
             "weight_decay": 0.0, "name": "lora"},
        ]
        param_groups = [g for g in param_groups if g["params"]]
        optimizer    = torch.optim.AdamW(param_groups)

        from transformers import get_cosine_schedule_with_warmup
        # train_loader    = self.trainer.fit_loop._data_source.dataloader()
        # steps_per_epoch = len(train_loader) // max(1, self.trainer.accumulate_grad_batches)
        steps_per_epoch = (8500 // self.args.batch_size) // 2
        total_steps     = steps_per_epoch * self.args.max_epochs
        warmup_steps    = steps_per_epoch * 1

        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )
        return {
            "optimizer":    optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "step", "frequency": 1},
        }

print("✅ HateClassifier (LiLT + WebContext, 4-modal) defined.")

✅ HateClassifier (LiLT + WebContext, 4-modal) defined.


In [69]:
seed_everything(42, workers=True)
# Pass web_context_lookup to both Dataset and Collator
collator = MemesCollator(args, web_context_lookup=web_context_lookup)

ds_train       = MemesDataset(args.dataset_root, "train",       args.image_size,
                               ocr_lookup=ocr_lookup, web_context_lookup=web_context_lookup)
ds_val         = MemesDataset(args.dataset_root, "dev_seen",    args.image_size,
                               ocr_lookup=ocr_lookup, web_context_lookup=web_context_lookup)
ds_test        = MemesDataset(args.dataset_root, "test_seen",   args.image_size,
                               ocr_lookup=ocr_lookup, web_context_lookup=web_context_lookup)
ds_val_unseen  = MemesDataset(args.dataset_root, "dev_unseen",  args.image_size,
                               ocr_lookup=ocr_lookup, web_context_lookup=web_context_lookup)
ds_test_unseen = MemesDataset(args.dataset_root, "test_unseen", args.image_size,
                               ocr_lookup=ocr_lookup, web_context_lookup=web_context_lookup)

NUM_WORKERS = 2

dl_train = DataLoader(ds_train, batch_size=args.batch_size, shuffle=True,
                      collate_fn=collator, num_workers=NUM_WORKERS,
                      pin_memory=True, persistent_workers=True, prefetch_factor=2)
dl_val        = DataLoader(ds_val,         batch_size=args.batch_size, shuffle=False,
                           collate_fn=collator, num_workers=NUM_WORKERS,
                           persistent_workers=True, prefetch_factor=2)
dl_test       = DataLoader(ds_test,        batch_size=args.batch_size, shuffle=False,
                           collate_fn=collator, num_workers=NUM_WORKERS,
                           persistent_workers=True, prefetch_factor=2)
dl_val_unseen = DataLoader(ds_val_unseen,  batch_size=args.batch_size, shuffle=False,
                           collate_fn=collator, num_workers=NUM_WORKERS,
                           persistent_workers=True, prefetch_factor=2)
dl_test_unseen= DataLoader(ds_test_unseen, batch_size=args.batch_size, shuffle=False,
                           collate_fn=collator, num_workers=NUM_WORKERS,
                           persistent_workers=True, prefetch_factor=2)

print(f"Train:{len(ds_train):,}  Val:{len(ds_val):,}  Test:{len(ds_test):,}  "
      f"Val-Unseen:{len(ds_val_unseen):,}  Test-Unseen:{len(ds_test_unseen):,}")

Train:8,500  Val:500  Test:1,000  Val-Unseen:540  Test-Unseen:2,000


In [70]:
run_name     = f"{args.name}_{args.dataset}_{args.experiment_name}"
log_csv_path = f"{args.experiment_log_dir}/{run_name}_epoch_log.csv"

epoch_logger = EpochLogger(log_csv_path, run_name)

ckpt_callback = ModelCheckpoint(
    dirpath          = args.checkpoint_dir,
    filename         = run_name + "-{epoch:02d}-{val/auroc:.4f}",
    monitor          = "val/auroc",
    mode             = "max",
    save_top_k       = 2,
    save_last        = True,
    save_weights_only= True,
    verbose          = True,
)
early_stop = EarlyStopping(
    monitor  = "val/auroc",
    patience = 5,
    mode     = "max",
    verbose  = True,
)

model = HateClassifier(args)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Parameter summary:")
print(f"   Total      : {total/1e6:.1f}M")
print(f"   Trainable  : {trainable/1e6:.1f}M")
print(f"   Frozen     : {(total-trainable)/1e6:.1f}M")
print(f"   Branches   : CLIP(frozen) + Phi(frozen) + LiLT-RoBERTa(frozen) + "
      f"LiLT-LoRA(trainable) + WebCtx(trainable) + proj/comb(trainable)")

trainer = Trainer(
    accelerator            = "gpu" if device == "cuda" else "cpu",
    devices                = 1,
    max_epochs             = args.max_epochs,
    gradient_clip_val      = args.gradient_clip_val,
    accumulate_grad_batches= 2,
    log_every_n_steps      = args.log_every_n_steps,
    val_check_interval     = args.val_check_interval,
    limit_train_batches    = args.limit_train_batches,
    limit_val_batches      = args.limit_val_batches,
    callbacks              = [ckpt_callback, early_stop, epoch_logger],
    deterministic          = False,
    enable_progress_bar    = True,
    precision              = "16-mixed",
)
print(f"\n🚀 Ready — run: {run_name}")

Loading weights:   0%|          | 0/400 [00:00<?, ?it/s]

   LiLT: Froze 124.65M text-side params
   LiLTLayoutBranch (LoRA r=4): 0.037M / 130.79M params trainable (0.03%)


✅ Loaded pre-trained ISSUES projection weights into image_map & text_map

📊 Parameter summary:
   Total      : 591.8M
   Trainable  : 20.1M
   Frozen     : 571.7M
   Branches   : CLIP(frozen) + Phi(frozen) + LiLT-RoBERTa(frozen) + LiLT-LoRA(trainable) + WebCtx(trainable) + proj/comb(trainable)

🚀 Ready — run: issues-layout_hmc_issues_lilt_webctx_v1


In [71]:
trainer.fit(model, train_dataloaders=dl_train, val_dataloaders=dl_val)
print(f"\n✅ Training complete.  Best checkpoint: {ckpt_callback.best_model_path}")

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


  📋  Epoch History — issues-layout_hmc_issues_lilt_webctx_v1
   Ep    |   TrLoss   |   VaLoss   |   TrAcc   |   VaAcc   |   TrF1    |   VaF1    |   TrAUROC   |   VaAUROC 
-------+------------+------------+-----------+-----------+-----------+-----------+-------------+-----------
★   1    |     —      |   0.9080   |    —      |    —      |    —      |  0.7841   |     —       |     —     
★   1    |   0.8853   |   0.9133   |    —      |  0.6981   |  0.5454   |  0.7561   |     —       |   0.7358  
  ★ = best val AUROC (epoch 1)



Validation: |          | 0/? [00:00<?, ?it/s]

... (32 outputs left out)


  📋  Epoch History — issues-layout_hmc_issues_lilt_webctx_v1
   Ep    |   TrLoss   |   VaLoss   |   TrAcc   |   VaAcc   |   TrF1    |   VaF1    |   TrAUROC   |   VaAUROC 
-------+------------+------------+-----------+-----------+-----------+-----------+-------------+-----------
    1    |     —      |   0.9080   |    —      |    —      |    —      |  0.7841   |     —       |     —     
    1    |   0.8853   |   0.9133   |    —      |  0.6981   |  0.5454   |  0.7561   |     —       |   0.7358  
    2    |   0.6854   |   0.8445   |  0.5696   |  0.7220   |  0.6761   |  0.7673   |   0.6590    |   0.8233  
    3    |   0.5232   |   0.8936   |  0.7426   |  0.7540   |  0.7607   |  0.7769   |   0.8291    |   0.8439  
    4    |   0.4724   |   0.8266   |  0.8171   |  0.7620   |  0.7849   |  0.7969   |   0.9058    |   0.8526  
    5    |   0.4219   |   0.8005   |  0.8362   |  0.7680   |  0.8119   |  0.7783   |   0.9241    |   0.8558  
    6    |   0.3715   |   0.8934   |  0.8587   |  0.7540   |

Validation: |          | 0/? [00:00<?, ?it/s]


  📋  Epoch History — issues-layout_hmc_issues_lilt_webctx_v1
   Ep    |   TrLoss   |   VaLoss   |   TrAcc   |   VaAcc   |   TrF1    |   VaF1    |   TrAUROC   |   VaAUROC 
-------+------------+------------+-----------+-----------+-----------+-----------+-------------+-----------
    1    |     —      |   0.9080   |    —      |    —      |    —      |  0.7841   |     —       |     —     
    1    |   0.8853   |   0.9133   |    —      |  0.6981   |  0.5454   |  0.7561   |     —       |   0.7358  
    2    |   0.6854   |   0.8445   |  0.5696   |  0.7220   |  0.6761   |  0.7673   |   0.6590    |   0.8233  
    3    |   0.5232   |   0.8936   |  0.7426   |  0.7540   |  0.7607   |  0.7769   |   0.8291    |   0.8439  
    4    |   0.4724   |   0.8266   |  0.8171   |  0.7620   |  0.7849   |  0.7969   |   0.9058    |   0.8526  
    5    |   0.4219   |   0.8005   |  0.8362   |  0.7680   |  0.8119   |  0.7783   |   0.9241    |   0.8558  
    6    |   0.3715   |   0.8934   |  0.8587   |  0.7540   |


✅ Training complete.  Best checkpoint: /kaggle/working/checkpoints/issues-layout_hmc_issues_lilt_webctx_v1-epoch=16-val/auroc=0.8768.ckpt


In [72]:
best_ckpt  = ckpt_callback.best_model_path
# best_ckpt = "/kaggle/working/checkpoints/..."   # override if needed

print(f"Loading best checkpoint: {best_ckpt}")
best_model = HateClassifier.load_from_checkpoint(best_ckpt, args=args)
best_model.eval().to(device)

def collect_predictions(model, dataloader, split_name):
    model.eval()
    all_preds, all_probs, all_labels, all_paths = [], [], [], []
    running_correct = 0; running_total = 0
    running_probs   = []; running_labels = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc=f"  {split_name}", ncols=100, leave=False)
        for batch in pbar:
            batch = {k: (v.to(device) if isinstance(v, torch.Tensor) else v)
                     for k, v in batch.items()}
            out   = model.common_step(batch)
            preds  = out["preds"].cpu().numpy()
            probs  = out["preds_proxy"].cpu().numpy()
            labels = out["labels"].cpu().numpy()
            all_preds.append(preds); all_probs.append(probs)
            all_labels.append(labels)
            all_paths.extend(batch.get("img_paths", [""] * len(preds)))
            valid = labels >= 0
            if valid.any():
                running_correct += (preds[valid] == labels[valid]).sum()
                running_total   += valid.sum()
                running_probs.extend(probs[valid].tolist())
                running_labels.extend(labels[valid].tolist())
            if running_total > 0:
                running_acc   = running_correct / running_total
                running_auroc = roc_auc_score(running_labels, running_probs) if len(set(running_labels)) > 1 else 0.5
            else:
                running_acc, running_auroc = 0.0, 0.5
            pbar.set_postfix({"Acc": f"{running_acc:.3f}", "AUROC": f"{running_auroc:.3f}"})

    all_labels = np.concatenate(all_labels)
    all_preds  = np.concatenate(all_preds)
    all_probs  = np.concatenate(all_probs)
    return all_labels, all_preds, all_probs, all_paths

results = {}
print("\nEvaluating splits:")
for split, dl in [("dev_seen", dl_val), ("test_seen", dl_test),
                  ("dev_unseen", dl_val_unseen), ("test_unseen", dl_test_unseen)]:
    lbl, pred, prob, paths = collect_predictions(best_model, dl, split)
    valid = lbl >= 0
    results[split] = {
        "labels": lbl[valid], "preds": pred[valid], "probs": prob[valid],
        "all_labels": lbl, "all_preds": pred, "all_probs": prob, "paths": paths,
    }
    if valid.sum() > 0:
        acc = (lbl[valid] == pred[valid]).mean()
        auc = roc_auc_score(lbl[valid], prob[valid])
        p, r, f, _ = precision_recall_fscore_support(lbl[valid], pred[valid], average="binary")
        print(f"  {split:15s}  Acc={acc:.4f}  AUROC={auc:.4f}  F1={f:.4f}  P={p:.4f}  R={r:.4f}")

print("\n✅ Evaluation complete.")

Loading best checkpoint: /kaggle/working/checkpoints/issues-layout_hmc_issues_lilt_webctx_v1-epoch=16-val/auroc=0.8768.ckpt


Loading weights:   0%|          | 0/400 [00:00<?, ?it/s]

   LiLT: Froze 124.65M text-side params
   LiLTLayoutBranch (LoRA r=4): 0.037M / 130.79M params trainable (0.03%)
✅ Loaded pre-trained ISSUES projection weights into image_map & text_map

Evaluating splits:


  dev_seen         Acc=0.7700  AUROC=0.8768  F1=0.7439  P=0.8267  R=0.6761


  test_seen        Acc=0.8030  AUROC=0.9042  F1=0.7814  P=0.8564  R=0.7184


  dev_unseen       Acc=0.8037  AUROC=0.8757  F1=0.7181  P=0.7670  R=0.6750


  test_unseen      Acc=0.8135  AUROC=0.8987  F1=0.7401  P=0.7752  R=0.7080

✅ Evaluation complete.


In [73]:
log_df = pd.read_csv(log_csv_path)
exp_summary = {
    "run_name":       run_name,
    "model":          args.name,
    "layout_model":   "LiLT (SCUT-DLVCLab/lilt-roberta-en-base)",
    "web_context":    args.use_web_context,
    "web_ctx_dim":    args.web_context_dim,
    "web_ctx_proj":   args.web_context_proj_dim,
    "map_dim":        args.map_dim,
    "layout_dim":     args.layout_dim,
    "lora_r":         args.lora_r,
    "lora_alpha":     args.lora_alpha,
    "lora_target":    args.lora_target_modules,
    "lilt_text_frozen": True,
    "phi_frozen":     True,
    "lr":             args.lr,
    "batch_size":     args.batch_size,
    "best_epoch":     int(log_df.loc[log_df["val_auroc"].dropna().idxmax(), "epoch"]),
    "results": {
        split: {
            "accuracy": float((res["labels"]==res["preds"]).mean()),
            "auroc":    float(roc_auc_score(res["labels"], res["probs"])),
            "ap":       float(average_precision_score(res["labels"], res["probs"])),
        } for split, res in results.items()
    },
    "epoch_log_csv": log_csv_path,
    "best_ckpt":     best_ckpt,
}

json_path = f"{args.experiment_log_dir}/{run_name}_summary.json"
with open(json_path, "w") as f:
    json.dump(exp_summary, f, indent=2)

print("\n" + "="*65)
print(f"  FINAL RESULTS — {run_name}")
print("="*65)
for split, metrics in exp_summary["results"].items():
    print(f"  {split:15s}  Acc={metrics['accuracy']:.4f}  "
          f"AUROC={metrics['auroc']:.4f}  AP={metrics['ap']:.4f}")
print("="*65)
print(f"  Best Epoch   : {exp_summary['best_epoch']}")
print(f"  Checkpoint   : {best_ckpt}")
print(f"  Web Context  : {args.use_web_context} (dim={args.web_context_dim}→{args.web_context_proj_dim})")
print(f"  Layout Model : LiLT (lilt-roberta-en-base)")
print(f"  LoRA rank    : r={args.lora_r}, alpha={args.lora_alpha}")
print("="*65)
print(f"✅ Summary → {json_path}")


  FINAL RESULTS — issues-layout_hmc_issues_lilt_webctx_v1
  dev_seen         Acc=0.7700  AUROC=0.8768  AP=0.8732
  test_seen        Acc=0.8030  AUROC=0.9042  AP=0.8978
  dev_unseen       Acc=0.8037  AUROC=0.8757  AP=0.8125
  test_unseen      Acc=0.8135  AUROC=0.8987  AP=0.8429
  Best Epoch   : 17
  Checkpoint   : /kaggle/working/checkpoints/issues-layout_hmc_issues_lilt_webctx_v1-epoch=16-val/auroc=0.8768.ckpt
  Web Context  : True (dim=768→256)
  Layout Model : LiLT (lilt-roberta-en-base)
  LoRA rank    : r=4, alpha=8
✅ Summary → /kaggle/working/experiments/issues-layout_hmc_issues_lilt_webctx_v1_summary.json


In [74]:
import shutil, os

output_dir = args.output_dir   # "/kaggle/working/outputs"
zip_path   = "/kaggle/working/outputs_results.zip"

shutil.make_archive(
    base_name = zip_path.replace(".zip", ""),
    format    = "zip",
    root_dir  = os.path.dirname(output_dir),
    base_dir  = os.path.basename(output_dir),
)

size_mb = os.path.getsize(zip_path) / 1e6
print(f"✅ Zipped → {zip_path}  ({size_mb:.1f} MB)")
print(f"   Files included:")
for f in sorted(os.listdir(output_dir)):
    fsize = os.path.getsize(os.path.join(output_dir, f)) / 1e6
    print(f"   {f:<45} {fsize:.2f} MB")

✅ Zipped → /kaggle/working/outputs_results.zip  (0.0 MB)
   Files included:


In [75]:
import os
import zipfile

folder_path = "/kaggle/working"
zip_path = "/kaggle/working/kaggle_output.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            
            # avoid zipping the zip itself
            if file_path == zip_path:
                continue
                
            arcname = os.path.relpath(file_path, folder_path)
            zipf.write(file_path, arcname)
            print("Added:", arcname)

print("ZIP completed:", zip_path)

Added: state.db
Added: outputs_results.zip
Added: .virtual_documents/__notebook_source__.ipynb
Added: labels/hmc_info.csv
Added: ocr_cache/ocr_full_dataset.pkl
Added: lightning_logs/version_2/hparams.yaml
Added: lightning_logs/version_2/events.out.tfevents.1780947921.aec1b3511ee7.58.2
Added: lightning_logs/version_1/events.out.tfevents.1780947565.aec1b3511ee7.58.1
Added: lightning_logs/version_1/hparams.yaml
Added: lightning_logs/version_0/hparams.yaml
Added: lightning_logs/version_0/events.out.tfevents.1780946018.aec1b3511ee7.58.0


KeyboardInterrupt: 

In [76]:
# ═══════════════════════════════════════════════════════════════════════
# CELL A — Research-paper-quality evaluation plots + reports
# ═══════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    roc_curve, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score,
    roc_auc_score, precision_recall_fscore_support,
)
from PIL import Image as PILImage
import warnings
warnings.filterwarnings("ignore")

os.makedirs(args.output_dir, exist_ok=True)

SPLITS      = ["dev_seen", "test_seen", "dev_unseen", "test_unseen"]
COLORS      = {"dev_seen": "#4C72B0", "test_seen": "#DD8452",
               "dev_unseen": "#55A868", "test_unseen": "#C44E52"}
SPLIT_LABEL = {"dev_seen": "Dev Seen", "test_seen": "Test Seen",
               "dev_unseen": "Dev Unseen", "test_unseen": "Test Unseen"}

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 12, "axes.labelsize": 11,
    "figure.dpi": 150, "axes.spines.top": False, "axes.spines.right": False,
})

# ═══════════════════════════════════════════════
# FIGURE 1 — ROC + PR curves side by side
# ═══════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle("Model Performance — ROC & Precision-Recall Curves",
             fontsize=14, fontweight="bold", y=1.02)

for split in SPLITS:
    res = results[split]
    lbl, prob = res["labels"], res["probs"]
    if len(set(lbl)) < 2: continue
    c = COLORS[split]; sl = SPLIT_LABEL[split]

    fpr, tpr, _ = roc_curve(lbl, prob)
    auc = roc_auc_score(lbl, prob)
    axes[0].plot(fpr, tpr, color=c, lw=2.2, label=f"{sl}  (AUC = {auc:.4f})")

    prec_c, rec_c, _ = precision_recall_curve(lbl, prob)
    ap = average_precision_score(lbl, prob)
    axes[1].plot(rec_c, prec_c, color=c, lw=2.2, label=f"{sl}  (AP = {ap:.4f})")

axes[0].plot([0,1],[0,1],"k--", lw=1.2, alpha=0.4, label="Random")
axes[0].fill_between([0,1],[0,1], alpha=0.04, color="gray")
axes[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate",
            title="ROC Curve", xlim=[0,1], ylim=[0,1.02])
axes[0].legend(fontsize=9.5, loc="lower right")
axes[0].grid(alpha=0.25)

baseline = (np.array([r["labels"] for r in results.values() if len(r["labels"])>0][0]) == 1).mean()
axes[1].axhline(baseline, color="gray", lw=1.2, ls="--", alpha=0.5, label=f"Random (P={baseline:.2f})")
axes[1].set(xlabel="Recall", ylabel="Precision",
            title="Precision-Recall Curve", xlim=[0,1], ylim=[0,1.05])
axes[1].legend(fontsize=9.5, loc="upper right")
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.savefig(f"{args.output_dir}/fig1_roc_pr.png", dpi=150, bbox_inches="tight")
plt.show(); print("✅ Figure 1 saved")

# ═══════════════════════════════════════════════
# FIGURE 2 — Confusion matrices (2×2 grid)
# ═══════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Confusion Matrices — All Splits",
             fontsize=14, fontweight="bold")

for ax, split in zip(axes.flatten(), SPLITS):
    res  = results[split]
    lbl, pred, prob = res["labels"], res["preds"], res["probs"]
    if len(set(lbl)) < 2: ax.axis("off"); continue

    cm   = confusion_matrix(lbl, pred)
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)   # row-normalised

    sns.heatmap(cm, annot=False, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Not Hateful","Hateful"],
                yticklabels=["Not Hateful","Hateful"],
                linewidths=0.5, linecolor="white", cbar=False)

    for i in range(2):
        for j in range(2):
            ax.text(j+0.5, i+0.35, f"{cm[i,j]}",
                    ha="center", va="center", fontsize=18, fontweight="bold",
                    color="white" if cm_n[i,j] > 0.6 else "black")
            ax.text(j+0.5, i+0.65, f"({cm_n[i,j]*100:.1f}%)",
                    ha="center", va="center", fontsize=10,
                    color="white" if cm_n[i,j] > 0.6 else "black")

    acc   = (lbl == pred).mean()
    auroc = roc_auc_score(lbl, prob)
    p, r, f, _ = precision_recall_fscore_support(lbl, pred, average="binary", zero_division=0)
    ax.set_title(
        f"{SPLIT_LABEL[split]}\n"
        f"Acc={acc:.4f}   AUROC={auroc:.4f}   F1={f:.4f}",
        fontsize=11, pad=10)
    ax.set_xlabel("Predicted Label"); ax.set_ylabel("True Label")

plt.tight_layout(h_pad=3, w_pad=2)
plt.savefig(f"{args.output_dir}/fig2_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show(); print("✅ Figure 2 saved")

# ═══════════════════════════════════════════════
# FIGURE 3 — Score distributions
# ═══════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Predicted Probability Distributions — Hateful vs Not Hateful",
             fontsize=14, fontweight="bold")

for ax, split in zip(axes.flatten(), SPLITS):
    res = results[split]
    lbl, prob = res["labels"], res["probs"]

    ax.hist(prob[lbl==0], bins=40, alpha=0.65, color="#4C72B0",
            label="Not Hateful", density=True, edgecolor="white", lw=0.3)
    ax.hist(prob[lbl==1], bins=40, alpha=0.65, color="#C44E52",
            label="Hateful",     density=True, edgecolor="white", lw=0.3)
    ax.axvline(0.5, color="black", linestyle="--", lw=1.8, label="Threshold = 0.5")

    auroc = roc_auc_score(lbl, prob) if len(set(lbl)) > 1 else 0
    ax.set_title(f"{SPLIT_LABEL[split]}  (AUROC={auroc:.4f})", fontsize=11)
    ax.set_xlabel("Predicted Probability of Hateful")
    ax.set_ylabel("Density")
    ax.legend(fontsize=9); ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(f"{args.output_dir}/fig3_score_dist.png", dpi=150, bbox_inches="tight")
plt.show(); print("✅ Figure 3 saved")

# ═══════════════════════════════════════════════
# FIGURE 4 — Per-split metric bar chart
# ═══════════════════════════════════════════════
metrics_data = {}
for split in SPLITS:
    res = results[split]
    lbl, pred, prob = res["labels"], res["preds"], res["probs"]
    if len(set(lbl)) < 2: continue
    p, r, f, _ = precision_recall_fscore_support(lbl, pred, average="binary", zero_division=0)
    metrics_data[SPLIT_LABEL[split]] = {
        "Accuracy":  (lbl == pred).mean(),
        "AUROC":     roc_auc_score(lbl, prob),
        "F1":        f,
        "Precision": p,
        "Recall":    r,
        "AP":        average_precision_score(lbl, prob),
    }

metric_names = ["Accuracy", "AUROC", "F1", "Precision", "Recall", "AP"]
split_names  = list(metrics_data.keys())
x = np.arange(len(metric_names))
bar_w = 0.18

fig, ax = plt.subplots(figsize=(15, 6))
for i, (split_label, vals) in enumerate(metrics_data.items()):
    bars = ax.bar(
        x + i * bar_w,
        [vals[m] for m in metric_names],
        bar_w, label=split_label,
        color=list(COLORS.values())[i], alpha=0.85,
        edgecolor="white", lw=0.5,
    )
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f"{h:.3f}", ha="center", va="bottom", fontsize=7.5, rotation=0)

ax.set_xticks(x + bar_w * (len(split_names)-1)/2)
ax.set_xticklabels(metric_names, fontsize=11)
ax.set_ylim(0, 1.08)
ax.set_ylabel("Score")
ax.set_title("Per-Split Performance Summary", fontsize=14, fontweight="bold")
ax.legend(fontsize=10, loc="upper right")
ax.grid(axis="y", alpha=0.25)
ax.axhline(1.0, color="gray", lw=0.8, ls="--", alpha=0.4)

plt.tight_layout()
plt.savefig(f"{args.output_dir}/fig4_metric_bars.png", dpi=150, bbox_inches="tight")
plt.show(); print("✅ Figure 4 saved")

# ═══════════════════════════════════════════════
# FIGURE 5 — Error analysis (FP + FN examples)
# ═══════════════════════════════════════════════
for split in ["dev_seen", "test_seen"]:
    res   = results[split]
    lbl   = res["labels"]
    pred  = res["preds"]
    prob  = res["probs"]
    paths = res["paths"]

    fp_idx = np.where((pred==1) & (lbl==0))[0]
    fn_idx = np.where((pred==0) & (lbl==1))[0]
    fp_idx = fp_idx[np.argsort(prob[fp_idx])[::-1]][:4]
    fn_idx = fn_idx[np.argsort(prob[fn_idx])][:4]

    fig = plt.figure(figsize=(16, 8))
    fig.suptitle(
        f"Error Analysis — {SPLIT_LABEL[split]}\n"
        "Top row: False Positives (predicted Hateful, actually Not)   "
        "Bottom row: False Negatives (predicted Not, actually Hateful)",
        fontsize=11, fontweight="bold")

    gs = gridspec.GridSpec(2, 4, hspace=0.45, wspace=0.25)

    for col, idx in enumerate(fp_idx):
        ax = fig.add_subplot(gs[0, col])
        try:
            img = PILImage.open(paths[idx]).convert("RGB")
            ax.imshow(img)
        except:
            ax.text(0.5,0.5,"No image",ha="center",va="center",transform=ax.transAxes)
        for sp in ax.spines.values():
            sp.set_edgecolor("#d62728"); sp.set_linewidth(3); sp.set_visible(True)
        ax.set_title(f"FP  P(hate)={prob[idx]:.3f}", fontsize=9, color="#d62728", pad=4)
        ax.axis("off")

    for col, idx in enumerate(fn_idx):
        ax = fig.add_subplot(gs[1, col])
        try:
            img = PILImage.open(paths[idx]).convert("RGB")
            ax.imshow(img)
        except:
            ax.text(0.5,0.5,"No image",ha="center",va="center",transform=ax.transAxes)
        for sp in ax.spines.values():
            sp.set_edgecolor("#e67e22"); sp.set_linewidth(3); sp.set_visible(True)
        ax.set_title(f"FN  P(hate)={prob[idx]:.3f}", fontsize=9, color="#e67e22", pad=4)
        ax.axis("off")

    plt.savefig(f"{args.output_dir}/fig5_errors_{split}.png", dpi=130, bbox_inches="tight")
    plt.show()
    print(f"✅ Figure 5 ({split}) saved")

# ═══════════════════════════════════════════════
# PRINT — Classification reports + summary table
# ═══════════════════════════════════════════════
print("\n" + "="*70)
print("  CLASSIFICATION REPORTS")
print("="*70)
for split in SPLITS:
    res = results[split]
    lbl, pred = res["labels"], res["preds"]
    if len(set(lbl)) < 2: continue
    print(f"\n── {SPLIT_LABEL[split]} ──")
    print(classification_report(lbl, pred,
          target_names=["Not Hateful","Hateful"], digits=4))

print("\n" + "="*80)
print(f"  {'Split':<15} {'Acc':>8} {'AUROC':>8} {'AP':>8} {'F1':>8} {'Prec':>8} {'Rec':>8}")
print("="*80)
for split in SPLITS:
    res = results[split]
    lbl, pred, prob = res["labels"], res["preds"], res["probs"]
    if len(set(lbl)) < 2: continue
    acc   = (lbl == pred).mean()
    auroc = roc_auc_score(lbl, prob)
    ap    = average_precision_score(lbl, prob)
    p, r, f, _ = precision_recall_fscore_support(lbl, pred, average="binary", zero_division=0)
    print(f"  {SPLIT_LABEL[split]:<15} {acc:>8.4f} {auroc:>8.4f} {ap:>8.4f} "
          f"{f:>8.4f} {p:>8.4f} {r:>8.4f}")
print("="*80)
print(f"\n✅ All figures saved to {args.output_dir}/")

<Figure size 2100x825 with 2 Axes>

✅ Figure 1 saved


<Figure size 1800x1500 with 4 Axes>

✅ Figure 2 saved


<Figure size 2100x1500 with 4 Axes>

✅ Figure 3 saved


... (2 outputs left out)

<Figure size 2400x1200 with 8 Axes>

✅ Figure 5 (dev_seen) saved


<Figure size 2400x1200 with 8 Axes>

✅ Figure 5 (test_seen) saved

  CLASSIFICATION REPORTS

── Dev Seen ──
              precision    recall  f1-score   support

 Not Hateful     0.7315    0.8617    0.7913       253
     Hateful     0.8267    0.6761    0.7439       247

    accuracy                         0.7700       500
   macro avg     0.7791    0.7689    0.7676       500
weighted avg     0.7786    0.7700    0.7679       500


── Test Seen ──
              precision    recall  f1-score   support

 Not Hateful     0.7657    0.8843    0.8207       510
     Hateful     0.8564    0.7184    0.7814       490

    accuracy                         0.8030      1000
   macro avg     0.8111    0.8013    0.8011      1000
weighted avg     0.8102    0.8030    0.8014      1000


── Dev Unseen ──
              precision    recall  f1-score   support

 Not Hateful     0.8214    0.8794    0.8494       340
     Hateful     0.7670    0.6750    0.7181       200

    accuracy                         0.8037       540
   macro avg     0.794

In [77]:
# ═══════════════════════════════════════════════
# FIGURE 5 — Correctly Predicted Random Samples
# ═══════════════════════════════════════════════
import random

LABEL_MAP = {
    0: "Not Hateful",
    1: "Hateful"
}

for split in ["dev_seen", "test_seen"]:
    res   = results[split]
    lbl   = np.array(res["labels"])
    pred  = np.array(res["preds"])
    prob  = np.array(res["probs"])
    paths = res["paths"]

    # Correct predictions only
    correct_idx = np.where(lbl == pred)[0]

    if len(correct_idx) == 0:
        print(f"No correct predictions found for {split}")
        continue

    # Randomly sample up to 8 images
    sample_count = min(8, len(correct_idx))
    sampled_idx = random.sample(list(correct_idx), sample_count)

    fig = plt.figure(figsize=(16, 8))
    fig.suptitle(
        f"Correctly Classified Samples — {SPLIT_LABEL[split]}",
        fontsize=13,
        fontweight="bold"
    )

    gs = gridspec.GridSpec(2, 4, hspace=0.35, wspace=0.25)

    for i, idx in enumerate(sampled_idx):
        ax = fig.add_subplot(gs[i // 4, i % 4])

        try:
            img = PILImage.open(paths[idx]).convert("RGB")
            ax.imshow(img)
        except:
            ax.text(
                0.5, 0.5, "No image",
                ha="center", va="center",
                transform=ax.transAxes
            )

        gt_label   = LABEL_MAP[int(lbl[idx])]
        pred_label = LABEL_MAP[int(pred[idx])]

        ax.set_title(
            f"GT: {gt_label}\nPred: {pred_label}\nP={prob[idx]:.3f}",
            fontsize=9,
            pad=5
        )

        # Green border for correct prediction
        for sp in ax.spines.values():
            sp.set_edgecolor("green")
            sp.set_linewidth(3)
            sp.set_visible(True)

        ax.axis("off")

    plt.savefig(
        f"{args.output_dir}/fig5_correct_samples_{split}.png",
        dpi=130,
        bbox_inches="tight"
    )

    plt.show()

    print(f"✅ Correct sample figure ({split}) saved")

<Figure size 2400x1200 with 8 Axes>

✅ Correct sample figure (dev_seen) saved


<Figure size 2400x1200 with 8 Axes>

✅ Correct sample figure (test_seen) saved


In [78]:
# ═══════════════════════════════════════════════
# FIGURE 5 — Qualitative Results (Correct + Misclassified)
# ═══════════════════════════════════════════════
import random

for split in ["dev_seen", "test_seen"]:

    res   = results[split]

    lbl   = np.array(res["labels"])
    pred  = np.array(res["preds"])
    prob  = np.array(res["probs"])
    paths = np.array(res["paths"])

    # Correct and incorrect indices
    correct_idx   = np.where(lbl == pred)[0]
    incorrect_idx = np.where(lbl != pred)[0]

    # Random samples
    num_correct   = min(6, len(correct_idx))
    num_incorrect = min(2, len(incorrect_idx))

    selected_correct   = np.random.choice(correct_idx, num_correct, replace=False)
    selected_incorrect = np.random.choice(incorrect_idx, num_incorrect, replace=False)

    selected_idx = np.concatenate([selected_correct, selected_incorrect])

    # Shuffle final display order
    np.random.shuffle(selected_idx)

    fig = plt.figure(figsize=(16, 8))
    fig.suptitle(
        f"Qualitative Results — {SPLIT_LABEL[split]}",
        fontsize=14,
        fontweight="bold"
    )

    gs = gridspec.GridSpec(2, 4, hspace=0.45, wspace=0.25)

    for i, idx in enumerate(selected_idx):

        ax = fig.add_subplot(gs[i // 4, i % 4])

        try:
            img = PILImage.open(paths[idx]).convert("RGB")
            ax.imshow(img)
        except:
            ax.text(
                0.5, 0.5, "No image",
                ha="center",
                va="center",
                transform=ax.transAxes
            )

        gt_label   = "Hateful" if lbl[idx] == 1 else "Not Hateful"
        pred_label = "Hateful" if pred[idx] == 1 else "Not Hateful"

        is_correct = lbl[idx] == pred[idx]

        # Green for correct, red for misclassified
        title_color  = "green" if is_correct else "red"
        border_color = "#2ca02c" if is_correct else "#d62728"

        ax.set_title(
            f"GT: {gt_label}\nPred: {pred_label}\nP(hate)={prob[idx]:.3f}",
            fontsize=9,
            color=title_color,
            fontweight="bold",
            pad=5
        )

        # Colored border
        for sp in ax.spines.values():
            sp.set_edgecolor(border_color)
            sp.set_linewidth(3)
            sp.set_visible(True)

        ax.axis("off")

    plt.savefig(
        f"{args.output_dir}/fig5_qualitative_{split}.png",
        dpi=140,
        bbox_inches="tight"
    )

    plt.show()

    print(f"✅ Figure 5 ({split}) saved")

<Figure size 2400x1200 with 8 Axes>

✅ Figure 5 (dev_seen) saved


<Figure size 2400x1200 with 8 Axes>

✅ Figure 5 (test_seen) saved


In [79]:
import torch, os, json, pickle
from pathlib import Path

SAVE_DIR = "/kaggle/working/saved_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── 1. Save state dict (weights only) ────────────────────────────────
torch.save(
    best_model.state_dict(),
    f"{SAVE_DIR}/model_weights.pt"
)
print(f"✅ Weights saved → {SAVE_DIR}/model_weights.pt  "
      f"({os.path.getsize(f'{SAVE_DIR}/model_weights.pt')/1e6:.1f} MB)")

# ── 2. Save full model (architecture + weights) ───────────────────────
# This lets you load with torch.load() without needing class definitions
torch.save(
    best_model,
    f"{SAVE_DIR}/model_full.pt"
)
print(f"✅ Full model saved → {SAVE_DIR}/model_full.pt  "
      f"({os.path.getsize(f'{SAVE_DIR}/model_full.pt')/1e6:.1f} MB)")

# ── 3. Save PL checkpoint (weights + optimizer + epoch + hparams) ─────
from pytorch_lightning.trainer.connectors.checkpoint_connector import (
    _CheckpointConnector,
)
pl_ckpt_path = f"{SAVE_DIR}/model_pl_checkpoint.ckpt"
trainer.save_checkpoint(pl_ckpt_path)
print(f"✅ PL checkpoint saved → {pl_ckpt_path}  "
      f"({os.path.getsize(pl_ckpt_path)/1e6:.1f} MB)")

# ── 4. Save args/config so inference script can rebuild Config ─────────
args_dict = vars(args).copy()
# drop non-serialisable fields
for k in list(args_dict.keys()):
    if not isinstance(args_dict[k], (str, int, float, bool, list)):
        args_dict.pop(k)
with open(f"{SAVE_DIR}/config.json", "w") as f:
    json.dump(args_dict, f, indent=2)
print(f"✅ Config saved → {SAVE_DIR}/config.json")

# ── 5. Save web context mean embedding (needed for inference fallback) ─
if web_context_lookup:
    import numpy as np
    all_embs = np.stack(list(web_context_lookup.values()))
    mean_emb = all_embs.mean(axis=0).astype(np.float32)
    mean_emb /= (np.linalg.norm(mean_emb) + 1e-8)
    np.save(f"{SAVE_DIR}/web_context_mean.npy", mean_emb)
    print(f"✅ Web context mean embedding saved → {SAVE_DIR}/web_context_mean.npy")

# ── 6. Summary ────────────────────────────────────────────────────────
print(f"\n📦 Saved model files:")
for f in sorted(os.listdir(SAVE_DIR)):
    fsize = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f"   {f:<40} {fsize:.2f} MB")

total = sum(os.path.getsize(os.path.join(SAVE_DIR,f))
            for f in os.listdir(SAVE_DIR)) / 1e6
print(f"\n   Total: {total:.1f} MB")

✅ Weights saved → /kaggle/working/saved_model/model_weights.pt  (2367.7 MB)
✅ Full model saved → /kaggle/working/saved_model/model_full.pt  (2368.1 MB)
✅ PL checkpoint saved → /kaggle/working/saved_model/model_pl_checkpoint.ckpt  (2521.9 MB)
✅ Config saved → /kaggle/working/saved_model/config.json
✅ Web context mean embedding saved → /kaggle/working/saved_model/web_context_mean.npy

📦 Saved model files:
   config.json                              0.00 MB
   model_full.pt                            2368.09 MB
   model_pl_checkpoint.ckpt                 2521.90 MB
   model_weights.pt                         2367.74 MB
   web_context_mean.npy                     0.00 MB

   Total: 7257.7 MB
